In [1]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
print("Project root added to sys.path")

Project root added to sys.path


In [2]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
from models.model import get_tinyprop_model
from models.config import get_tinyprop_config
from clients.federated_training import federated_training
from utils.data_partition import dirichlet_partition
import matplotlib.pyplot as plt
import os


c:\Users\la7tim\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: '[WinError 127] The specified procedure could not be found'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [3]:
import os

# Create results directory in the notebook directory
os.makedirs("results", exist_ok=True)
print("Results directory created at:", os.path.abspath("results"))

Results directory created at: c:\Users\la7tim\Desktop\Internship\FedTinyProp\notebooks\results


In [4]:
# Define transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Load CIFAR-10
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Print dataset info
print(f"Training set size: {len(trainset)}")
print(f"Test set size: {len(testset)}")
print(f"Number of classes: {len(trainset.classes)}")

Files already downloaded and verified
Files already downloaded and verified
Training set size: 50000
Test set size: 10000
Number of classes: 10


In [5]:
num_clients = 5
partitions = {
    "dirichlet_iid":      dirichlet_partition(trainset, num_clients=num_clients, alpha=1000.0),  # High alpha -> IID-like
    "dirichlet_noniid":   dirichlet_partition(trainset, num_clients=num_clients, alpha=0.01),   # Low alpha -> non-IID
    "dirichlet_medium":   dirichlet_partition(trainset, num_clients=num_clients, alpha=0.5),   # Medium skew
}

In [6]:
# Get model configuration
config = get_tinyprop_config("cifar10")
print("\nModel Configuration:")
for key, value in config.items():
    print(f"{key}: {value}")

# Initialize model parameters
tinyprop_params = config["tinyprop_params"]
print("\nTinyProp Parameters:")
print(f"S_min: {tinyprop_params.S_min}")
print(f"S_max: {tinyprop_params.S_max}")
print(f"zeta: {tinyprop_params.zeta}")
print(f"number_of_layers: {tinyprop_params.number_of_layers}")


Model Configuration:
tinyprop_params: <models.tinyProp.TinyPropParams object at 0x0000023C16484310>
skip_threshold: 0.1
full_flops_per_batch: 3000000.0
phi_min: 0.2
optimizer: {'type': 'sgd', 'lr': 0.1, 'momentum': 0.9, 'weight_decay': 0.0005, 'nesterov': True}
lr_scheduler: {'type': 'cosine', 'T_max': 100, 'eta_min': 0.0001, 'warmup_epochs': 5, 'warmup_start_lr': 0.01}
gradient_clip: 0.5
batch_size: 128
num_epochs: 2
label_smoothing: 0.1
data_augmentation: {'random_crop': True, 'random_horizontal_flip': True, 'random_rotation': 5, 'color_jitter': {'brightness': 0.2, 'contrast': 0.2, 'saturation': 0.2}}

TinyProp Parameters:
S_min: 0.05
S_max: 0.3
zeta: 0.2
number_of_layers: 3


In [7]:
from clients.aggregators import sparse_fedavg_aggregate
# First, let's create a function to run training for each partition
def train_and_analyze_partition(partition_name, client_datasets, tinyprop_params):
    print(f"\nTraining on partition: {partition_name.upper()}")
    
    # Run training
    model, accuracy_list, flops_list, mem_list, comm_list, sparsity_list, \
    avg_grad_norm_list, avg_phi_list, skipped_batches_list, \
    effective_compute_ratio_list, client_eval_history, compression_ratio_list, \
    history = federated_training(
        client_datasets=client_datasets,
        model_name='cifar10',
        testset=testset,
        tinyprop_params=tinyprop_params,
        aggregator_fn=sparse_fedavg_aggregate,
        rounds=100,
        device="cuda" if torch.cuda.is_available() else "cpu",
        local_epochs=1,
        early_stopping_patience=100,
        early_stopping_delta=0.001,
        csv_log_path=f'results/cifar10_{partition_name}_training_log.csv',
        initial_sparsity=tinyprop_params.S_min,
        target_sparsity=tinyprop_params.S_max,
        energy_budget=1000
    )
    
    return model, history

partition_results = {}
for strategy_name, client_datasets in partitions.items():
    print(f"\nStarting training for {strategy_name.upper()} partition...")
    try:
        model, history = train_and_analyze_partition(strategy_name, client_datasets, tinyprop_params)
        partition_results[strategy_name] = {
            'model': model,
            'history': history
        }
        print(f"Completed training for {strategy_name.upper()}")
    except Exception as e:
        print(f"Error in {strategy_name.upper()}: {str(e)}")
        continue



Starting training for DIRICHLET_IID partition...

Training on partition: DIRICHLET_IID

Round 1/100
Epoch 1/1, Batch 7/79, Loss: 7.0745, LR: 0.064000


c:\Users\la7tim\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\optim\lr_scheduler.py:243: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


Epoch 1/1, Batch 14/79, Loss: 5.0179, LR: 0.099901
Epoch 1/1, Batch 21/79, Loss: 3.4369, LR: 0.099606
Epoch 1/1, Batch 28/79, Loss: 3.0651, LR: 0.098431
Epoch 1/1, Batch 35/79, Loss: 2.7691, LR: 0.097047
Epoch 1/1, Batch 42/79, Loss: 2.5970, LR: 0.095246
Epoch 1/1, Batch 49/79, Loss: 2.6725, LR: 0.094556
Epoch 1/1, Batch 56/79, Loss: 2.5546, LR: 0.092224
Epoch 1/1, Batch 63/79, Loss: 2.5690, LR: 0.091363
Epoch 1/1, Batch 70/79, Loss: 2.5460, LR: 0.090460
Epoch 1/1, Batch 77/79, Loss: 2.5783, LR: 0.089518
Epoch 1/1, Batch 7/79, Loss: 6.9778, LR: 0.064000
Epoch 1/1, Batch 14/79, Loss: 4.9786, LR: 0.099901
Epoch 1/1, Batch 21/79, Loss: 3.7599, LR: 0.099606
Epoch 1/1, Batch 35/79, Loss: 2.7203, LR: 0.097555
Epoch 1/1, Batch 49/79, Loss: 2.6153, LR: 0.095892
Epoch 1/1, Batch 63/79, Loss: 2.5596, LR: 0.093044
Epoch 1/1, Batch 77/79, Loss: 2.6371, LR: 0.091363
Epoch 1/1, Batch 7/79, Loss: 7.4203, LR: 0.064000
Epoch 1/1, Batch 14/79, Loss: 5.1373, LR: 0.099901
Epoch 1/1, Batch 21/79, Loss: 3.6

KeyboardInterrupt: 